In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import altair as alt
import pyarrow
import json
from datetime import datetime

In [7]:
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [2]:
df_hora = pd.read_csv('data/dados_hora.csv')
df_diarios = pd.read_csv('data/dados_diarios.csv')

In [3]:
df_hora['Resto (kWh)'] = df_hora['Rede Distribuição (kWh)'] - (df_hora['Eólica (kWh)'] + df_hora['Fotovoltaica (kWh)'] + df_hora['Hídrica (kWh)'] )

In [4]:
dfIII = pd.read_csv('data/dados_producao_mesano.csv')

In [5]:
df_hora.describe()

,Cogeração (kWh),Eólica (kWh),Fotovoltaica (kWh),Hídrica (kWh),Outras Tecnologias (kWh),Rede Distribuição (kWh),Baixa Tensão (kWh),Média Tensão (kWh),Alta Tensão (kWh),Muito Alta Tensão (kWh),Dia,Mês,Ano,Mercado (kWh),Regime Especial (kWh),Total (kWh) (Consumido),Total (kWh) (Produzido),Resto (kWh)
count,109981.000000,1.099810e+05,109981.000000,109981.000000,109981.000000,1.099810e+05,1.099810e+05,109981.000000,109981.000000,109981.000000,109981.000000,109981.000000,109981.000000,1.099810e+05,1.099810e+05,1.099810e+05,1.099810e+05,109981.000000
mean,42107.022813,3.839374e+05,9399.109369,23595.893400,65396.774071,5.244362e+05,7.622991e+05,452311.478722,197598.470101,71292.390699,15.661051,6.298706,2024.084687,9.604390e+05,5.244274e+05,1.483501e+06,1.484866e+06,107503.796884
std,25470.863331,2.830175e+05,13271.313071,17321.372333,23767.494822,2.832140e+05,2.141918e+05,120512.717902,19256.419943,17242.062001,8.804726,3.533235,0.895114,3.616615e+05,2.832203e+05,2.786397e+05,2.783659e+05,26403.338227
min,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,1.000000,1.000000,2023.000000,-2.267620e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
25%,19103.750000,1.496928e+05,0.250000,6168.500000,43854.468000,2.944831e+05,6.084690e+05,350009.521600,187234.566600,59152.361900,8.000000,3.000000,2023.000000,7.246130e+05,2.944607e+05,1.252330e+06,1.253547e+06,90500.750000
50%,47480.750000,3.193232e+05,183.250000,21362.000000,62524.546000,4.612802e+05,7.273177e+05,432361.067400,199737.935800,72896.314800,16.000000,6.000000,2024.000000,9.711140e+05,4.612802e+05,1.473203e+06,1.474199e+06,106883.789000
75%,63679.250000,5.651491e+05,18112.750000,40631.250000,83214.500000,7.011767e+05,8.786290e+05,554676.295600,210840.052400,85661.813000,23.000000,9.000000,2025.000000,1.214783e+06,7.011767e+05,1.668029e+06,1.669938e+06,123777.750000
max,102841.000000,1.238478e+06,50510.500000,95408.000000,144804.805006,1.466503e+06,1.729600e+06,761619.116200,244902.224000,109840.462400,31.000000,12.000000,2026.000000,2.178261e+06,1.466503e+06,2.561410e+06,2.561410e+06,201650.053023


In [8]:
tecnologias = ['Eólica (kWh)', 'Fotovoltaica (kWh)', 'Hídrica (kWh)', 'Resto (kWh)']

cores = px.colors.qualitative.Set3[:4] 

# Criar o mapeamento (domain = nomes, range = cores)
color_scale = alt.Scale(domain=tecnologias, range=cores)

# 2. "Derreter" o DataFrame
df_long = df_hora.melt(
    id_vars=['Data/Hora'], 
    value_vars=tecnologias,
    var_name='Tecnologia', 
    value_name='kWh'
)

# 3. Criar o gráfico
chart = alt.Chart(df_long).mark_area().encode(
    alt.X('yearmonth(Data/Hora):T').axis(format='%b %Y', title='Mês/Ano'),
    alt.Y('sum(kWh):Q').title('Produção Total (kWh)'),
    
    # Adicionamos o 'sort' aqui para ordenar pela soma de kWh
    alt.Color('Tecnologia:N').scale(color_scale),
    
    alt.Order('sum(kWh):Q', sort='descending'),
    
    tooltip=['yearmonth(Data/Hora)', 'Tecnologia', 'sum(kWh)']
).properties(
    width=800,
    height=400,
    title='Evolução Mensal da Produção de Energia'
).interactive()

chart.show()

alt.Chart(...)

In [9]:
cores_dic = {k:v for k,v in zip(tecnologias,cores)}

In [26]:
dfII = df_hora.copy()
dfII = dfII.groupby(by=[dfII['Ano'],dfII['Mês']]).sum().reset_index()
dfII = dfII[['Ano', 'Mês', 'Eólica (kWh)', 'Fotovoltaica (kWh)', 'Hídrica (kWh)', 'Resto (kWh)']]
dfII['Ano'] = dfII['Ano'].astype(str)
dfII['Mes/Ano'] = dfII['Mês'].astype(str) + '/' + dfII['Ano'].astype(str)
dfII = dfII.iloc[:-2,:]
anos_legend= list(dfII.apply(
    lambda x: str(x['Ano']) if x['Mês'] == 6 else "", axis=1))

def create_circular_histogram(tec:str):
    # ... (teu código inicial de criação do fig e dfII igual) ...
    r = tec
    if r not in dfII.columns:
        raise ValueError(f"Tecnologia '{r}' não encontrada.")
    
    # Garantir que anos_legend está atualizado
    anos_legend[-1] = "2026"

    fig = px.bar_polar(
        dfII,
        r=r,
        theta="Mes/Ano",
        color="Ano",
        template="plotly_white",
        color_discrete_sequence=px.colors.qualitative.D3
    ).update_layout(
        showlegend=True,
        coloraxis_showscale=False,
        legend=dict(
            title="Ano de Produção",
            font=dict(size=12),
            # Isto garante que a legenda não fica preta se o fundo for escuro
            itemsizing='constant' 
        ),
        polar=dict(hole = 0.2,
            angularaxis=dict(
                    type="category",
                    # IMPORTANTE: O array de categorias tem de ser a coluna theta completa
                    categoryarray=dfII['Mes/Ano'].tolist(),
                    categoryorder="array",
                    # O período tem de ser o número total de fatias para fechar o círculo
                    period=len(dfII),
                    tickvals=dfII['Mes/Ano'].tolist(),
                    # O texto é que leva a lista com vazios
                    ticktext=anos_legend,
                    direction="clockwise",
                    rotation=90,
                    tickfont=dict(size=18, family='Arial', style='italic')
            ),  
            radialaxis=dict(
                showticklabels=True, 
                range=[0, dfII[r].max()*1.02],
                nticks=5, 
                tickfont=dict(size=18, family='Arial'),
                linecolor='black', 
                linewidth=1,
                layer='above traces', # Mudei para 'below' para as barras não taparem os números
                gridcolor='lightgrey'
            ),   
        ),
        
        height=600,
        width=800,
        margin=dict(b=30, t=80, l=0, r=0),
        )

    # --- LÓGICA DAS LINHAS DIVISÓRIAS (CORRIGIDA) ---
    max_r = dfII[r].max()
    
    for i in range(-1,len(dfII) - 1):
        ano_atual = dfII.iloc[i]['Ano']
        ano_proximo = dfII.iloc[i+1]['Ano']
        
        if ano_atual != ano_proximo:
            # Pegamos no nome da categoria atual (Dezembro) 
            # e na categoria seguinte (Janeiro)
            cat_dez = dfII.iloc[i]['Mes/Ano']
            cat_jan = dfII.iloc[i+1]['Mes/Ano']
            
            # Adicionamos a linha usando os nomes das categorias
            # O Plotly desenha a linha na transição entre estas duas
            fig.add_trace(go.Scatterpolar(
                r=[0, max_r * 1.1],
                # Usar a categoria de Janeiro como ponto de referência
                theta=[cat_jan, cat_jan], 
                mode='lines',
                line=dict(color='black', width=2, dash='dash'),
                hoverinfo='none',
                showlegend=False
            ))
    return fig
"""
    # --- REAJUSTE DO LAYOUT PARA NÃO CRIAR ESPAÇOS ---
    fig.update_layout(
        polar=dict(
            angularaxis=dict(
                type="category",
                categoryarray=dfII['Mes/Ano'].tolist(),
                categoryorder="array",
                # Garante que não há espaços extras
                period=len(dfII['Mes/Ano'].unique()), 
                # ... resto das tuas configs ...
            )
        )
    )
    fig.update_layout(
        title=f"Evolução Mensal da Produção de Energia: {r}",
        showlegend=True,
        polar=dict(
            hole=0.2,
            bgcolor='white',
            angularaxis=dict(
                type="category",
                categoryarray=dfII['Mes/Ano'].tolist(),
                categoryorder="array",
                period=len(dfII),
                tickvals=dfII['Mes/Ano'].tolist(),
                ticktext=anos_legend,
                direction="clockwise",
                rotation=90,
                tickfont=dict(size=14, family='Arial', style='italic'),
                gridcolor='lightgrey'
            ),
            radialaxis=dict(
                gridcolor='lightgrey',
                showticklabels=True,
                tickfont=dict(size=12, color='black', family='Arial Black'),
                layer='above traces'
            )
        ),
        legend=dict(
            itemclick=False, 
            itemdoubleclick=False,
            title="Ano de Produção"
        ),
        height=700,
        width=800,
        margin=dict(b=50, t=80, l=50, r=50)
    )"""
    
fig = create_circular_histogram('Eólica (kWh)')
fig.show()

In [27]:
def create_circular_histogram(tec:str):
    r = tec
    if r not in dfII.columns:
        raise ValueError(f"Tecnologia '{r}' não encontrada.")
    
    #anos_legend[-1] = "2026"


    # 1. Criar a figura base
    fig = px.bar_polar(
        dfII,
        r=r,
        theta="Mes/Ano",
        color="Ano",
        template="plotly_white",
        color_discrete_sequence=px.colors.qualitative.D3
    )

    # 2. Adicionar as linhas divisórias pretas
    

    # 3. CONFIGURAÇÃO FINAL (Forçar Hole e Grelha)
    fig.update_layout(
        title=f"Evolução Mensal da Produção de Energia: {r}",
        height=700,
        width=800,
        polar=dict(
            hole=0.2,          # RECOLOCADO AQUI para garantir que aparece
            bgcolor='white',   # Fundo do circulo branco
            angularaxis=dict(
                type="category",
                categoryarray=dfII['Mes/Ano'].tolist(),
                categoryorder="array",
                period=len(dfII),
                tickvals=dfII['Mes/Ano'].tolist(),
                ticktext=anos_legend,
                direction="clockwise",
                rotation=90,
                gridcolor='lightgrey', # COR DA GRELHA (Linhas radiais)
                showgrid=True,
                tickfont=dict(size=14, family='Arial', style='italic')
            ),
            radialaxis=dict(
                gridcolor='lightgrey', # COR DA GRELHA (Círculos)
                showgrid=True,
                showticklabels=True,
                tickfont=dict(size=12, color='black', family='Arial Black'),
                layer='above traces'
            )
        ),
        legend=dict(
            itemclick=False, 
            itemdoubleclick=False,
            title="Ano de Produção"
        ),
        margin=dict(b=50, t=80, l=50, r=50)
    )
    max_r = dfII[r].max()
    for i in range(-1, len(dfII) - 1):
        if dfII.iloc[i]['Ano'] != dfII.iloc[i+1]['Ano']:
            cat_jan = dfII.iloc[i+1]['Mes/Ano']
            fig.add_trace(go.Scatterpolar(
                r=[0, max_r * 1.1],
                theta=[cat_jan, cat_jan], 
                mode='lines',
                line=dict(color='black', width=2, dash='dash'),
                hoverinfo='none',
                showlegend=False
            ))
    
    return fig
fig = create_circular_histogram('Eólica (kWh)')
fig.show()

In [29]:
def create_circular_histogram(tec:str):
    r = tec
    if r not in dfII.columns:
        raise ValueError(f"Tecnologia '{r}' não encontrada.")
    
    # 1. Criar a figura base
    # Usamos template=None para ter controlo total sobre as cores da grelha
    fig = px.bar_polar(
        dfII,
        r=r,
        theta="Mes/Ano",
        color="Ano",
        template=None, 
        color_discrete_sequence=px.colors.qualitative.D3
    )

    # 2. Adicionar as linhas divisórias entre anos
    # Estas são adicionadas como traces de Scatterpolar
    max_r = dfII[r].max()
    for i in range(len(dfII) - 1):
        if dfII.iloc[i]['Ano'] != dfII.iloc[i+1]['Ano']:
            # A categoria de Janeiro do ano seguinte serve como o ponto da linha
            cat_transicao = dfII.iloc[i+1]['Mes/Ano']
            
            fig.add_trace(go.Scatterpolar(
                r=[0, max_r * 1.1],
                theta=[cat_transicao, cat_transicao], 
                mode='lines',
                # Cor cinza com transparência (0.4) para não "cortar" as barras agressivamente
                line=dict(color='rgba(80, 80, 80, 0.4)', width=2, dash='dash'),
                hoverinfo='none',
                showlegend=False
            ))

    # 3. Configuração do Layout (Grelha, Hole e Legenda)
    fig.update_layout(
        title=f"Evolução Mensal da Produção de Energia: {r}",
        paper_bgcolor='white',
        plot_bgcolor='white',
        height=700,
        width=800,
        polar=dict(
            hole=0.2,
            bgcolor='white',
            angularaxis=dict(
                type="category",
                categoryarray=dfII['Mes/Ano'].tolist(),
                categoryorder="array",
                period=len(dfII),
                tickvals=dfII['Mes/Ano'].tolist(),
                ticktext=anos_legend,
                direction="clockwise",
                rotation=90,
                gridcolor='lightgrey', # Linhas que saem do centro
                showgrid=True,
                tickfont=dict(size=14, family='Arial', style='italic')
            ),
            radialaxis=dict(
                gridcolor='lightgrey', # Círculos de escala
                showgrid=True,
                showticklabels=True,
                tickfont=dict(size=12, color='black', family='Arial Black'),
                # 'above traces' faz com que a escala (0.2B, 0.4B...) fique por cima de tudo
                layer='above traces' 
            )
        ),
        legend=dict(
            itemclick=False, 
            itemdoubleclick=False,
            title="Ano de Produção",
            orientation="v",
            x=1.1 # Move a legenda para fora do círculo
        ),
        margin=dict(b=50, t=80, l=50, r=100)
    )
    
    return fig
fig = create_circular_histogram('Eólica (kWh)')
fig.show()

In [5]:
dfII = df_hora.copy()
dfII = dfII.groupby(by=[dfII['Ano'],dfII['Mês']]).sum().reset_index()
dfII = dfII[['Ano', 'Mês', 'Eólica (kWh)', 'Fotovoltaica (kWh)', 'Hídrica (kWh)', 'Resto (kWh)']]
dfII['Ano'] = dfII['Ano'].astype(str)
dfII['Mes/Ano'] = dfII['Mês'].astype(str) + '/' + dfII['Ano'].astype(str)
anos_legend= list(dfII.apply(
    lambda x: str(x['Ano']) if x['Mês'] == 6 else "", axis=1
))
def create_circular_histogram(tec:str):
    r = tec
    if r not in dfII.columns:
        raise ValueError(f"Tecnologia '{r}' não encontrada. Opções: {dfII.columns[2:-1].tolist()}")
    anos_legend[-1] = "2026"


    fig = go.Figure()
    max_r = dfII[r].max() 

    #.add_trace(go.Scatterpolar(
    #    r=[0, max_r * 1.1],
    #    theta=[-0.5, -0.5],
    #    mode='lines',
    #    line=dict(color='grey', width=3),
    #    hoverinfo='none',
    #    showlegend=False
    #))
    ponto_partida = dfII['Mes/Ano'].iloc[0]
    ponto_final = dfII['Mes/Ano'].iloc[-2]
    ponto_meio = dfII['Mes/Ano'].iloc[len(dfII)//2]
    print(ponto_partida)
    max_r = dfII[r].max() 


    fig.add_trace(go.Scatterpolar(
        r=[0,max_r * 1.1],
        theta=[ponto_meio,ponto_meio], 
        thetaunit = "degrees",
        mode='lines',
        line=dict(color='black', width=2, dash='dash'),
        hoverinfo='none',
        showlegend=False
    ))


    fig_temp = px.bar_polar(
        dfII,
        r= r,
        theta="Mes/Ano",
        color="Ano",
        labels="Ano",
        title=f"Evolução Mensal da Produção de Energia: {r}",
        color_discrete_sequence=px.colors.qualitative.D3
    ).update_layout(
        showlegend=True,
        coloraxis_showscale=False,
        legend=dict(
            title="Ano de Produção",
            font=dict(size=12),
            # Isto garante que a legenda não fica preta se o fundo for escuro
            itemsizing='constant' 
        ),
        polar=dict(hole = 0.2,
            angularaxis=dict(
                    type="category",
                    # IMPORTANTE: O array de categorias tem de ser a coluna theta completa
                    categoryarray=dfII['Mes/Ano'].tolist(),
                    categoryorder="array",
                    # O período tem de ser o número total de fatias para fechar o círculo
                    period=len(dfII),
                    tickvals=dfII['Mes/Ano'].tolist(),
                    # O texto é que leva a lista com vazios
                    ticktext=anos_legend,
                    direction="clockwise",
                    rotation=90,
                    tickfont=dict(size=18, family='Arial', style='italic')
            ),  
            
             
        ),
        
        height=600,
        width=800,
        margin=dict(b=30, t=80, l=0, r=0),
        )

    for trace in fig_temp.data:
        fig.add_trace(trace)

    fig.show()
create_circular_histogram('Eólica (kWh)')

1/2023


In [ ]:
dfII = df_hora.copy()
dfII = dfII.groupby(by=[dfII['Ano'],dfII['Mês']]).sum().reset_index()
dfII = dfII[['Ano', 'Mês', 'Eólica (kWh)', 'Fotovoltaica (kWh)', 'Hídrica (kWh)', 'Resto (kWh)']]
dfII['Ano'] = dfII['Ano'].astype(str)
dfII['Mes/Ano'] = dfII['Mês'].astype(str) + '/' + dfII['Ano'].astype(str)
anos_legend= list(dfII.apply(
    lambda x: str(x['Ano']) if x['Mês'] == 6 else "", axis=1
))
def create_circular_histogram(tec:str):
    r = tec
    if r not in dfII.columns:
        raise ValueError(f"Tecnologia '{r}' não encontrada. Opções: {dfII.columns[2:-1].tolist()}")
    anos_legend[-1] = "2026"
    fig = px.bar_polar(
        dfII,
        r= r,
        theta="Mes/Ano",
        color="Ano",
        labels="Ano",
        title=f"Evolução Mensal da Produção de Energia: {r}",
        color_discrete_sequence=px.colors.qualitative.D3
    ).update_layout(
        showlegend=True,
        coloraxis_showscale=False,
        legend=dict(
            title="Ano de Produção",
            font=dict(size=12),
            # Isto garante que a legenda não fica preta se o fundo for escuro
            itemsizing='constant' 
        ),
        polar=dict(hole = 0.2,
            angularaxis=dict(
                    type="category",
                    # IMPORTANTE: O array de categorias tem de ser a coluna theta completa
                    categoryarray=dfII['Mes/Ano'].tolist(),
                    categoryorder="array",
                    # O período tem de ser o número total de fatias para fechar o círculo
                    period=len(dfII),
                    tickvals=dfII['Mes/Ano'].tolist(),
                    # O texto é que leva a lista com vazios
                    ticktext=anos_legend,
                    direction="clockwise",
                    rotation=90,
                    tickfont=dict(size=18, family='Arial', style='italic')
            ),  
            radialaxis=dict(
                showticklabels=True, 
                range=[0, dfII[r].max()*1.02],
                nticks=5, 
                tickfont=dict(size=18, family='Arial'),
                linecolor='black', 
                linewidth=1,
                layer='above traces', # Mudei para 'below' para as barras não taparem os números
                gridcolor='lightgrey'
            ),   
        ),
        
        height=600,
        width=800,
        margin=dict(b=30, t=80, l=0, r=0),
        )
    max_r = dfII[r].max() 


    return fig 
fig = create_circular_histogram('Eólica (kWh)')
fig.show()

    

In [76]:
dfII = df_hora.copy()
dfII = dfII.groupby(by=[dfII['Ano'],dfII['Mês']]).sum().reset_index()
dfII = dfII[['Ano', 'Mês', 'Eólica (kWh)', 'Fotovoltaica (kWh)', 'Hídrica (kWh)', 'Resto (kWh)']]
dfII['Ano'] = dfII['Ano'].astype(str)
dfII['Mes/Ano'] = dfII['Mês'].astype(str) + '/' + dfII['Ano'].astype(str)
anos_legend= list(dfII.apply(
    lambda x: str(x['Ano']) if x['Mês'] == 6 else "", axis=1
))
import re
import colorsys
def ajustar_intensidade(rgb, fator):
    canais = re.findall(r'\d+', rgb)
    r, g, b = [int(c) for c in canais]
    
    # 2. Converter RGB (0-255) para HLS (0.0-1.0)
    h, l, s = colorsys.rgb_to_hls(r/255.0, g/255.0, b/255.0)
    
    # 3. Ajustar a luminosidade (Lightness)
    # fator > 1 clareia, fator < 1 escurece
    l = max(0, min(1, l * fator))
    
    # 4. Converter de volta para o formato string "rgb(r,g,b)"
    r_new, g_new, b_new = [round(x * 255) for x in colorsys.hls_to_rgb(h, l, s)]
    return f"rgb({r_new},{g_new},{b_new})"


def create_circular_histogram(tec:str):
    r = tec
    cor = cores_dic[tec]
    if r not in dfII.columns:
        raise ValueError(f"Tecnologia '{r}' não encontrada. Opções: {dfII.columns[2:-1].tolist()}")
    anos_legend[-1] = "2026"
    n_anos = dfII['Ano'].nunique()
    cores = [ajustar_intensidade(cor,1-(f*0.3)) for f in range(n_anos)]
    fig = px.bar_polar(
        dfII,
        r= r,
        theta="Mes/Ano",
        color="Ano",
        labels="Ano",
        title=f"Evolução Mensal da Produção de Energia: {r}",
        #color_discrete_sequence=px.colors.sequential.Teal_r
        color_discrete_sequence=cores
    ).update_layout(
        showlegend=True,
        coloraxis_showscale=False,
        legend=dict(
            title="Ano de Produção",
            font=dict(size=12),
            # Isto garante que a legenda não fica preta se o fundo for escuro
            itemsizing='constant' 
        ),
        polar=dict(hole = 0.2,
            angularaxis=dict(
                    type="category",
                    # IMPORTANTE: O array de categorias tem de ser a coluna theta completa
                    categoryarray=dfII['Mes/Ano'].tolist(),
                    categoryorder="array",
                    # O período tem de ser o número total de fatias para fechar o círculo
                    period=len(dfII),
                    tickvals=dfII['Mes/Ano'].tolist(),
                    # O texto é que leva a lista com vazios
                    ticktext=anos_legend,
                    direction="clockwise",
                    rotation=90,
                    tickfont=dict(size=18, family='Arial', style='italic')
            ),  
            radialaxis=dict(
                showticklabels=True, 
                range=[0, dfII[r].max()*1.02],
                nticks=5, 
                tickfont=dict(size=18, family='Arial'),
                linecolor='black', 
                linewidth=1,
                layer='above traces', # Mudei para 'below' para as barras não taparem os números
                gridcolor='lightgrey'
            ),   
        ),
        
        height=600,
        width=800,
        margin=dict(b=30, t=80, l=0, r=0),
        )
    max_r = dfII[r].max() 


    return fig 
fig = create_circular_histogram('Eólica (kWh)')
fig.show()


In [77]:
fig = create_circular_histogram('Hídrica (kWh)')
fig.show()

In [78]:
fig = create_circular_histogram('Fotovoltaica (kWh)')
fig.show()

In [79]:
fig = create_circular_histogram('Resto (kWh)')
fig.show()

In [7]:
tecnologias = ['Eólica (kWh)', 'Fotovoltaica (kWh)', 'Hídrica (kWh)', 'Resto (kWh)']

# O melt mantém 'Ano' e 'Mês' e transforma as colunas de tecnologia em linhas
df_new = df_hora.melt(
    id_vars=['Ano', 'Mês'], 
    value_vars=tecnologias,
    var_name='Tecnologia', 
    value_name='Producao'
)

dfIII = df_new.groupby(by=[df_new['Ano'],df_new['Mês'],df_new['Tecnologia']]).sum().reset_index()

dfIII['Ano'] = dfIII['Ano'].astype(str)
dfIII['Mes/Ano'] = dfIII['Mês'].astype(str) + '/' + dfIII['Ano'].astype(str)

# Adicionar espaço vazio entre a última e primeira barra para sinalizar o início
espaco_df = pd.DataFrame({
    'Ano': ['2026'],
    'Mês': [0],
    'Tecnologia': ['Espaço'],
    'Producao': [0],
    'Mes/Ano': ['0/2026']
})
dfIII = pd.concat([dfIII, espaco_df], ignore_index=True)

r = "Producao"
meses = dict(zip(range(1,13), ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']))
anos_legend= list(dfIII.apply(
    lambda x: meses[x['Mês']]+ "/" + x['Ano'] if x['Mês'] %2 != 0 else "", axis=1
))
#anos_legend[-1] = "2026"
fig = px.bar_polar(
    dfIII,
    r= r,
    theta="Mes/Ano",
    color="Tecnologia",
    labels="Ano",
    title=f"Evolução Mensal da Produção de Energia: {r}",
    color_discrete_sequence=px.colors.qualitative.Set3).update_layout(
    showlegend=True,
    coloraxis_showscale=False,
    legend=dict(
        title="Ano de Produção",
        font=dict(size=12),
        # Isto garante que a legenda não fica preta se o fundo for escuro
        itemsizing='constant' 
    ),
    polar=dict(hole = 0.2,
          angularaxis=dict(
                type="category",
                # IMPORTANTE: O array de categorias tem de ser a coluna theta completa
                categoryarray=dfIII['Mes/Ano'].tolist(),
                categoryorder="array",
                # O período tem de ser o número total de fatias para fechar o círculo
                period=len(dfIII['Mes/Ano'].unique()),
                tickvals=dfIII['Mes/Ano'].tolist(),
                # O texto é que leva a lista com vazios
                ticktext=anos_legend,
                direction="clockwise",
                rotation=90,
                tickfont=dict(size=18, family='Arial', style='italic')
        ),  
        radialaxis=dict(
            showticklabels=True, 
            range=[0, dfIII[r].max()*1.02],
            nticks=5, 
            tickfont=dict(size=18, family='Arial'),
            linecolor='black', 
            linewidth=1,
            layer='above traces', # Mudei para 'below' para as barras não taparem os números
            gridcolor='lightgrey'
        ),   
    ),
    
    height=1000,
    width=1000,
    margin=dict(b=30, t=80, l=100, r=60),
    )

fig.show()

In [21]:
dfIII.to_csv('data/dados_producao_mesano.csv', index=False)

In [186]:
def create_spiral_histogram(tec:str):
    if tec not in dfIII['Tecnologia'].unique():
        df_spiral = pd.DataFrame(columns=dfIII.columns)
        i = 0
        for mes,ano in dfIII[['Mês','Ano']].drop_duplicates().values:
            total = dfIII[(dfIII['Ano']==ano) & (dfIII['Mês']==mes)].copy()
            total = total.loc[:,'Producao'].sum()
            df_spiral.loc[i,:] = [ano,mes,'Total',total,str(mes)+'/'+str(ano)]
            i+=1
    else:
        df_spiral = dfIII[dfIII['Tecnologia'] == tec]
    df_spiral = df_spiral[['Ano', 'Mês', 'Producao']]
    max_val = df_spiral['Producao'].max()
    df_spiral = df_spiral.sort_values(['Ano', 'Mês']).reset_index(drop=True)
    df_spiral = df_spiral[:-1]
    df_spiral = df_spiral[:-1]

    # r_base faz com que cada mês suba um pouco, criando a espiral
    df_spiral['r_base'] = np.arange(len(df_spiral)) * 20
    df_spiral['r_top'] = df_spiral['r_base'] + (df_spiral['Producao'] / max_val * 5)

    # Normalizamos a produção para que o tamanho das barras seja proporcional
    # mas caiba bem na espiral.
    norm_factor = df_spiral['Producao'].max() * 0.05

    # Criamos a base da barra que aumenta continuamente com o tempo (a espiral)
    # Isto faz com que Dezembro e Janeiro do ano seguinte não se sobreponham.
    df_spiral['r_base'] = np.arange(len(df_spiral)) 
    df_spiral['r_top'] = df_spiral['r_base'] + (df_spiral['Producao'] / norm_factor)

    r_total_max = df_spiral['r_top'].max()

    # --- 2. Criação do Gráfico ---
    fig = go.Figure()
    cores_anos = {
    2023: "#636EFA", # Azul (Centro)
        2024: "#00CC96", # Verde (Meio)
        2025: "#EC9026", # Laranja (Exterior)
        "Total": "#AB63FA"}


    for year in df_spiral['Ano'].unique():
        temp = df_spiral[df_spiral['Ano'] == year]
    
        # 1. Calcular métricas e formatar strings (podes usar a função format_energy que sugeri antes)
        v_min = temp['Producao'].min()
        v_max = temp['Producao'].max()
        
        # Criar o nome detalhado para a legenda
        # .2g é mais amigável que .2e, mas usa o que preferires
        legend_name=f"<b>Ano {year}</b><br>Mín: {v_min}<br>Máx: {v_max}"
        
        # Pegar na cor uma única vez
        cor_do_ano = cores_anos.get(year, "grey")

        # 2. ADICIONAR APENAS UM TRACE POR ANO
        fig.add_trace(go.Barpolar(
            r=temp['Producao'] / norm_factor,
            theta=temp['Mês'] * (360/12), 
            base=temp['r_base'] * 2,
            name=legend_name,               # Usamos logo o nome detalhado aqui
            marker=dict(color=cor_do_ano),
            customdata=temp['Producao'],
            hovertemplate=(
                f"<b>Ano: {year}</b><br>"
                "Mês: %{theta}°<br>"
                "Produção: %{customdata:.3e} kWh"
                "<extra></extra>"
            ),
            thetaunit="degrees"
        ))


        """
        temp = df_spiral[df_spiral['Ano'] == year]
        
        cor_do_ano = cores_anos.get(year, "grey")

        fig.add_trace(go.Barpolar(
            r=temp['Producao'] / norm_factor, # Comprimento da barra
            # Convertemos meses (1-12) em graus (0-360) para ocupar o círculo todo
            theta=temp['Mês'] * (360/12), 
            base=temp['r_base']*2,          # O SEGREDO: O offset que cria a espiral
            name=str(year),
            customdata=temp['Producao'],
            hovertemplate="<b>Produção:</b> %{customdata:.3e} kWh<extra></extra>",
            thetaunit="degrees",
            marker=dict(color=cor_do_ano),
            
        ))
        v_min = temp['Producao'].min()
        v_max = temp['Producao'].max()
        
        # 2. Criar uma string de nome personalizada para a legenda
        # Usamos format .2s para converter 1.000.000 em 1M, por exemplo
        legend_name = f"<b>{year}</b> (Mín: {v_min:.2e} | Máx: {v_max:.2e})"
        
        fig.add_trace(go.Barpolar(
            r=temp['Producao'] / norm_factor,
            theta=temp['Mês'] * (360/12), 
            base=temp['r_base']*2,
            name=legend_name, # Aqui a legenda passa a ter a info extra
            marker=dict(color=cores_anos.get(year, "grey")),
            customdata=temp['Producao'],
            hovertemplate="<b>Ano:</b> " + str(year) + "<br><b>Produção:</b> %{customdata:.2e} kWh<extra></extra>",
            thetaunit="degrees"
        ))"""

    # --- 3. Layout Estilizado (Limpo e Focado na Forma) ---
    r_real_max = (df_spiral['r_base'].max() * 2) + (df_spiral['Producao'].max() / norm_factor)
    fig.update_layout(
        font_size=16,
        polar=dict(
            hole=0.2,
            angularaxis=dict(
                # Configuração para que os nomes dos meses apareçam nos graus certos
                tickvals=[30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360],
                ticktext=['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez'],
                direction="clockwise",
                rotation=120, # Janeiro no topo
                tickfont=dict(size=20, family='Arial', style='italic')
            ),
            # Removemos os círculos e números do eixo radial para ficar limpo
            
            radialaxis=dict(
                showticklabels=False,
                ticks="",
                showline=False,
                range=[0, r_real_max * 1.05] # Dá espaço para as últimas barras
            )
        ),
        # Mostramos a legenda apenas para os anos
        showlegend=True,
        legend=dict(
        itemclick=False,      # Desativa o clique simples (esconder/mostrar)
        itemdoubleclick=False # Desativa o duplo clique (isolar um ano)
    ),
        height=650,
        width=800,
        margin=dict(b=30, t=80, l=20, r=20),
    )

    fig.update_layout(
    legend=dict(
        orientation="v",       # Vertical para ler melhor os intervalos
        yanchor="middle",
        y=0.8,
        xanchor="left",
        x=1,                 # Afasta um pouco do gráfico
        font=dict(size=15),
        itemsizing='constant',   # Mantém os ícones uniformes
        itemwidth=30
    ),
    margin=dict(r=100)         # Aumenta a margem para a legenda caber
)

    fig.show()

In [187]:
create_spiral_histogram('Eólica (kWh)') 

In [32]:
def area_month_year_interval(month:int,year:int,interval:str):
    if month not in list(range(1,13)):
        raise ValueError('Mês deve estar no intervalo [1,12]')
    if year not in [2023,2024,2025,2026]:
        raise ValueError('Ano deve ser 2023,2024,2025 ou 2026!')
    if interval not in ['15m','1h','4h','12h','1d']:
        raise ValueError("Interval deve ser um de: ['15m','1h','4h','12h','1d']")
    dic_month = {1:'janeiro',2:'fevereiro',3:'março',4:'abril',5:'maio',6:'junho',7:'julho',8:'agosto',9:'setembro',10:'outubro',
                 11:'novembro',12:'dezembro'}
    df_group = df_hora[(df_hora['Ano']==year) & (df_hora['Mês']==month)].copy()
    if interval in ['1h','4h','12h']:
        horas = df_group['Data/Hora'].str.split('T').str[1].str.split(':').str[0].astype(int).rename('Hora_int')
        if interval in ['4h','12h']:
            inter = int(interval[:-1])
            """
            for hora in range(len(horas)):
                val = hora
                first = None
                if val%inter==0:
                    first = val
                else:
                    hora = first"""
            horas = (horas // inter) * inter
        df_group = df_group.groupby(by=[df_group['Dia'],horas]).sum()
        series = pd.Series([datetime(year,month,day,hour) for day,hour in df_group.index ])
    elif interval == '1d':
        df_group = df_group.groupby(by=df_group['Dia']).sum()
        series = pd.Series([datetime(year,month,day) for day in df_group.index ])

    else:
        horas = df_group['Data/Hora'].str.split('T').str[1].str.split(':').str[0].astype(int).rename('Hora_int')
        minutos = df_group['Data/Hora'].str.split('T').str[1].str.split(':').str[1].astype(int).rename('Minutos_int') 
        df_group = df_group.groupby(by=[df_group['Dia'],horas,minutos]).sum()
        series = pd.Series([datetime(year,month,day,hour,minute) for day,hour,minute in df_group.index ])
    
    df_group = df_group.reset_index()
    df_group = df_group[['Eólica (kWh)','Fotovoltaica (kWh)','Hídrica (kWh)','Resto (kWh)']]
    df_group['Hora'] = series
    tecnologias = ['Eólica (kWh)', 'Fotovoltaica (kWh)', 'Hídrica (kWh)', 'Resto (kWh)']
    df_group.sort_values(by='Hora')
# O melt mantém 'Ano' e 'Mês' e transforma as colunas de tecnologia em linhas
    df_new = df_group.melt(
    id_vars=['Hora'], 
    value_vars=tecnologias,
    var_name='Tecnologia', 
    value_name='Producao'
)
    fig = px.area(df_new, x="Hora", y="Producao", color="Tecnologia", 
                  title=f"Evolução da produção energética em {dic_month[month]} de {year}", 
                  #color_discrete_sequence=["#14FDA4", "#D6D30B", "#2375F0","grey"])
                  color_discrete_sequence=px.colors.qualitative.Set3)
    fig.show()

    
    

In [33]:
area_month_year_interval(8,2024,'1d')

In [34]:
area_month_year_interval(8,2024,'12h')

In [35]:
area_month_year_interval(8,2024,'4h')

In [36]:
area_month_year_interval(8,2024,'1h')

In [37]:
area_month_year_interval(8,2024,'15m')